# Script for Extracting Radios 

This script is for extracting the team radios of the 2023 season.

Races extracted:
* Belgium.
* Singapore.
* Spain.
* Monaco.
* Brazil.
* Netherlands.

### Flag for knowing if I needed to add more data due to eliminating post-race radios

I needed to add the last 3 GPs after eliminating post-race radios.

---

#### Getting meeting key
https://api.openf1.org/v1/meetings?year=2023&country_name=Spain
#### Getting session key
GET https://api.openf1.org/v1/sessions?meeting_key=1218&session_name=Race
#### Getting radios
https://api.openf1.org/v1/team_radio?session_key=9158&driver_number=11



In [5]:
"""
OpenF1 Team Radio Data Extraction Script - Multiple Grand Prix
------------------------------------------------------------------
Extracts all team radio data from the OpenF1 API for several GPs
and saves them in Parquet format as well as audio files.
"""

# ── Step 0 · Setup ────────────────────────────────────────────────────────────
import sys
import warnings
from pathlib import Path
import time
import requests
import pandas as pd

warnings.filterwarnings("ignore")

repo_root = Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── Paths ──────────────────────────────────────────────────────────────────────
# Aligning with N18 and N17 naming conventions
RAW_DIR   = repo_root / "data" / "raw"
AUDIO_DIR = repo_root / "data" / "raw" / "radio_audio"

for d in [RAW_DIR, AUDIO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"repo_root : {repo_root}")
print(f"RAW_DIR   : {RAW_DIR}")
print(f"AUDIO_DIR : {AUDIO_DIR}")

# Define the session_keys and names of the GPs we want to extract
gp_data = [
    {"name": "Spain", "session_key": 9158, "year": 2023},
    {"name": "Singapore", "session_key": 9165, "year": 2023},
    {"name": "Belgium", "session_key": 9141, "year": 2023},
    {"name": "Monaco", "session_key": 9094, "year": 2023},
    {"name": "Brazil", "session_key": 9205, "year": 2023},
    {"name": "Netherlands", "session_key": 9149, "year": 2023},
]

repo_root : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager
RAW_DIR   : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\raw
AUDIO_DIR : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\raw\radio_audio


In [6]:
def fetch_team_radio(session_key, gp_name):
    """
    Extracts ALL team radio messages for a specific race,
    without filtering by driver.
    """
    # The driver_number parameter is omitted to retrieve all radio messages
    url = f"https://api.openf1.org/v1/team_radio?session_key={session_key}"
    print(f"Fetching: {url}")
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        print(f"✓ Found {len(data)} records for GP {gp_name}")
        df = pd.DataFrame(data)
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
        
        # Add a column with the GP name for identification
        df['gp_name'] = gp_name
        
        return df
    except Exception as e:
        print(f"Error retrieving data for GP {gp_name}: {e}")
        return pd.DataFrame()


In [7]:
def download_radio_files(df, gp_name, audio_dir=AUDIO_DIR):
    """
    Downloads audio files for a specific team radio DataFrame.
    """
    print(f"\nDownloading audio files for the {gp_name} GP...")

    # Group by driver - passing the column name as string to avoid tuple keys like (1,)
    grouped = df.groupby("driver_number")

    total_downloads = 0
    for driver_number, group in grouped:
        # Ensure we have a clean string for the driver number (e.g. "1" instead of "1.0")
        driver_clean = str(driver_number).replace('.0', '')
        
        folder_name = f"driver_{driver_clean}"
        driver_folder = audio_dir / folder_name
        driver_folder.mkdir(parents=True, exist_ok=True)

        # Download and save audio files
        for i, row in group.iterrows():
            url = row["recording_url"]
            if pd.isna(url):
                continue

            # Create a filename including the GP name
            filename = f"driver_{driver_clean}_{gp_name.lower()}_radio_{i}.mp3"
            output_path = driver_folder / filename

            # Check if the file already exists to avoid duplicate downloads
            if output_path.exists():
                #print(f"File already exists: {output_path}")
                continue

            # Download the file
            try:
                response = requests.get(url)
                response.raise_for_status()
                with open(output_path, "wb") as f:
                    f.write(response.content)
                print(f"Downloaded: {output_path}")
                total_downloads += 1

                # Small delay to avoid overloading the server
                time.sleep(0.5)
            except Exception as e:
                print(f"Error downloading {url}: {e}")

    return total_downloads

In [8]:
# Extract data and download audio files for all defined GPs
all_dfs = []
total_files_downloaded = 0

for gp in gp_data:
    print(f"\n--- Processing {gp['name']} GP {gp['year']} ---")

    # Extract team radio data
    df_team_radio = fetch_team_radio(gp['session_key'], gp['name'])

    if not df_team_radio.empty:
        # Save as Parquet
        parquet_path = RAW_DIR / f"{gp['name']}_{gp['year']}_openf1_team_radio.parquet"
        df_team_radio.to_parquet(parquet_path)
        print(f"Data saved to {parquet_path}")

        # Download audio files
        files_downloaded = download_radio_files(df_team_radio, gp['name'])
        total_files_downloaded += files_downloaded

        # Add to the combined DataFrame
        all_dfs.append(df_team_radio)
    else:
        print(f"No data found for {gp['name']} GP")


--- Processing Spain GP 2023 ---
Fetching: https://api.openf1.org/v1/team_radio?session_key=9158
✓ Found 29 records for GP Spain
Data saved to c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\raw\Spain_2023_openf1_team_radio.parquet

Downloaded: c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\raw\radio_audio\driver_1\driver_1_spain_radio_5.mp3
Downloaded: c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\raw\radio_audio\driver_1\driver_1_spain_radio_9.mp3
Downloaded: c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\raw\radio_audio\driver_1\driver_1_spain_radio_14.mp3
Downloaded: c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\raw\radio_audio\driver_4\driver_4_spain_radio_11.mp3
Downloaded: c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\raw\radio_audio\driver_4\driver_4_spain_radio_13.mp3
Downloaded: c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Ma

In [9]:
# Combine all DataFrames into a single one
if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_path = RAW_DIR / "Combined_2023_openf1_team_radio.parquet"
    combined_df.to_parquet(combined_path)
    print(f"\n✓ All combined data saved to {combined_path}")

    # Display final statistics
    print(f"\n--- FINAL SUMMARY ---")
    print(f"Total GPs processed: {len(all_dfs)} out of {len(gp_data)}")
    print(f"Total team radio records: {len(combined_df)}")
    print(f"Total audio files downloaded: {total_files_downloaded}")

    # Show communication count per driver
    print("\nDistribution of communications per driver:")
    driver_counts = combined_df['driver_number'].value_counts().sort_index()
    for driver_num, count in driver_counts.items():
        print(f"  • Driver #{driver_num}: {count} communications")
else:
    print("No data found for any GP. Check your connection or session_keys.")


✓ All combined data saved to c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\raw\Combined_2023_openf1_team_radio.parquet

--- FINAL SUMMARY ---
Total GPs processed: 6 out of 6
Total team radio records: 659
Total audio files downloaded: 659

Distribution of communications per driver:
  • Driver #1: 55 communications
  • Driver #2: 17 communications
  • Driver #3: 7 communications
  • Driver #4: 33 communications
  • Driver #10: 24 communications
  • Driver #11: 38 communications
  • Driver #14: 38 communications
  • Driver #16: 27 communications
  • Driver #18: 19 communications
  • Driver #20: 25 communications
  • Driver #21: 7 communications
  • Driver #22: 16 communications
  • Driver #23: 26 communications
  • Driver #24: 26 communications
  • Driver #27: 37 communications
  • Driver #31: 38 communications
  • Driver #40: 7 communications
  • Driver #44: 48 communications
  • Driver #55: 65 communications
  • Driver #63: 62 communications
  • Driver #77: 22 c